# 실험 제목
- 담당: 김영빈
- 날짜: 26/09/21
- 목적: json 데이터를 csv파일 하나로 합치기

> 끝나면 결과를 `experiments/LOG.md`에 한 줄 남기기

## 1단계. 라이브러리 및 경로 설정

JSON을 읽고 표 형태로 변환하기 위해 `json`, `pathlib`, `pandas`를 사용한다. 이후 전체 파일을 처리할 때 진행률을 확인할 수 있도록 `tqdm`도 불러온다.

노트북을 저장소 루트 또는 `playground/youngbeen`에서 실행해도 같은 데이터 경로를 사용하도록 현재 위치에서 프로젝트 루트를 탐색한다. 입력 JSON과 생성할 CSV는 모두 Git 추적에서 제외된 `data/processed` 아래에 둔다. 현재 데이터는 training 데이터이므로 split 컬럼은 생성하지 않는다.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


def find_project_root(start_path: Path) -> Path:
    """현재 위치부터 상위 폴더를 확인해 프로젝트 루트를 찾는다."""
    start_path = start_path.resolve()

    for candidate in (start_path, *start_path.parents):
        if (candidate / ".git").exists() and (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. dontalk 저장소 내부에서 노트북을 실행해 주세요."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "qa_flat_train.csv"

if not PROCESSED_DIR.is_dir():
    raise NotADirectoryError(f"데이터 디렉토리가 없습니다: {PROCESSED_DIR}")

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"입력 디렉토리: {PROCESSED_DIR}")
print(f"출력 CSV: {OUTPUT_PATH}")
print(f"pandas 버전: {pd.__version__}")

## 2단계. 전체 JSON 파일 탐색 및 개수 확인

`data/processed` 아래의 모든 하위 디렉토리를 재귀적으로 탐색해 `.json` 파일 목록을 만든다. 은행·보험·증권의 폴더 번호를 코드에 직접 나열하지 않으므로, 새로운 하위 폴더가 추가되어도 자동으로 포함된다.

파일 목록은 항상 같은 순서로 처리할 수 있도록 정렬한다. 이 단계에서는 JSON 내부 내용은 아직 읽지 않고, 발견한 파일 수와 최상위 데이터 폴더별 개수만 확인한다.

In [ ]:
from collections import Counter


json_files = sorted(
    path for path in PROCESSED_DIR.rglob("*.json")
    if path.is_file()
)

if not json_files:
    raise FileNotFoundError(f"JSON 파일을 찾지 못했습니다: {PROCESSED_DIR}")

folder_counts = Counter(
    path.relative_to(PROCESSED_DIR).parts[0]
    for path in json_files
)

for folder_name, count in sorted(folder_counts.items()):
    print(f"[{folder_name}] {count:,}개")

print(f"\n총 JSON 파일 개수: {len(json_files):,}개")
print(f"첫 번째 파일: {json_files[0].relative_to(PROCESSED_DIR)}")
print(f"마지막 파일: {json_files[-1].relative_to(PROCESSED_DIR)}")

## 관찰 / 메모
